In [3]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('all')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package basque_gr

True

In [4]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [5]:
# Text preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert non-string types to string
    if not isinstance(text, str):
        text = str(text)
    # Remove punctuations and numbers
    text = re.sub('[^a-zA-Z]', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Remove stopwords and lemmatize
    text = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(text)

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Load the dataset
data = pd.read_csv('Suicide_Ideation_Dataset(Twitter-based).csv')
data['Tweet'] = data['Tweet'].apply(preprocess_text)

# Split the dataset into training and testing sets
train, test = train_test_split(data, test_size=0.2)

In [9]:
# Convert labels to float32 type
label_mapping = {'Potential Suicide post ': 1, 'Not Suicide post': 0}
train['Suicide'] = train['Suicide'].map(label_mapping).astype('float32')
test['Suicide'] = test['Suicide'].map(label_mapping).astype('float32')


# Tokenize the text
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train['Tweet'])
X_train = tokenizer.texts_to_sequences(train['Tweet'])
X_test = tokenizer.texts_to_sequences(test['Tweet'])

# Pad the sequences so they're all the same length
X_train = pad_sequences(X_train, maxlen=100)
X_test = pad_sequences(X_test, maxlen=100)

In [10]:
# Create the model
model = Sequential()
model.add(Embedding(5000, 64, input_length=100))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train, train['Suicide'], epochs=10, batch_size=32)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, test['Suicide'])
print('Test accuracy:', accuracy)

Epoch 1/10
45/45 [==============================] - 5s 53ms/step - loss: 0.6039 - accuracy: 0.6676
Epoch 2/10
45/45 [==============================] - 2s 54ms/step - loss: 0.3193 - accuracy: 0.8908
Epoch 3/10
45/45 [==============================] - 2s 51ms/step - loss: 0.0885 - accuracy: 0.9741
Epoch 4/10
45/45 [==============================] - 6s 129ms/step - loss: 0.0371 - accuracy: 0.9902
Epoch 5/10
45/45 [==============================] - 5s 111ms/step - loss: 0.0294 - accuracy: 0.9909
Epoch 6/10
45/45 [==============================] - 5s 101ms/step - loss: 0.0201 - accuracy: 0.9930
Epoch 7/10
45/45 [==============================] - 6s 129ms/step - loss: 0.0122 - accuracy: 0.9965
Epoch 8/10
45/45 [==============================] - 3s 62ms/step - loss: 0.0083 - accuracy: 0.9986
Epoch 9/10
45/45 [==============================] - 2s 53ms/step - loss: 0.0066 - accuracy: 0.9986
Epoch 10/10
12/12 [==============================] - 1s 23ms/step - loss: 0.2545 - accuracy: 0.9330
Test 

In [22]:
# Let's say we have a new tweet
new_tweet = "Sometime its better to end ourselves"

# Preprocess the tweet
new_tweet = preprocess_text(new_tweet)

# Tokenize and pad the tweet
new_tweet = tokenizer.texts_to_sequences([new_tweet])
new_tweet = pad_sequences(new_tweet, maxlen=100)

# Use the model to predict the sentiment of the tweet
prediction = model.predict(new_tweet)

# Print the prediction
if prediction > 0.5:
    print("This tweet is potentially suicidal.")
else:
    print("This tweet is not suicidal.")


1/1 [==============================] - 0s 30ms/step
This tweet is potentially suicidal.


In [23]:
# Hyperparameter Tuning
from keras.optimizers import Adam

# Create the model
model = Sequential()
model.add(Embedding(5000, 64, input_length=100))
model.add(LSTM(128))  # Increase the number of neurons
model.add(Dense(1, activation='sigmoid'))

# Compile the model with a custom learning rate
opt = Adam(learning_rate=0.01)
model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])

# Train the model with a larger batch size and more epochs
model.fit(X_train, train['Suicide'], epochs=10, batch_size=64)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, test['Suicide'])
print('Test accuracy:', accuracy)


Epoch 1/10
23/23 [==============================] - 8s 194ms/step - loss: 0.5598 - accuracy: 0.7369
Epoch 2/10
23/23 [==============================] - 4s 177ms/step - loss: 0.0996 - accuracy: 0.9706
Epoch 3/10
23/23 [==============================] - 7s 326ms/step - loss: 0.0252 - accuracy: 0.9902
Epoch 4/10
23/23 [==============================] - 6s 253ms/step - loss: 0.0093 - accuracy: 0.9958
Epoch 5/10
23/23 [==============================] - 4s 175ms/step - loss: 0.0036 - accuracy: 0.9993
Epoch 6/10
23/23 [==============================] - 6s 282ms/step - loss: 0.0026 - accuracy: 0.9993
Epoch 7/10
23/23 [==============================] - 4s 174ms/step - loss: 0.0032 - accuracy: 0.9986
Epoch 8/10
23/23 [==============================] - 4s 177ms/step - loss: 0.0014 - accuracy: 0.9993
Epoch 9/10
23/23 [==============================] - 5s 240ms/step - loss: 0.0017 - accuracy: 0.9993
Epoch 10/10
12/12 [==============================] - 1s 40ms/step - loss: 0.3185 - accuracy: 0.9330


In [25]:
# Let's say we have a new tweet
new_tweet = "End life button."

# Preprocess the tweet
new_tweet = preprocess_text(new_tweet)

# Tokenize and pad the tweet
new_tweet = tokenizer.texts_to_sequences([new_tweet])
new_tweet = pad_sequences(new_tweet, maxlen=100)

# Use the model to predict the sentiment of the tweet
prediction = model.predict(new_tweet)

# Print the prediction
if prediction > 0.5:
    print("This tweet is potentially suicidal.")
else:
    print("This tweet is not suicidal.")


1/1 [==============================] - 0s 29ms/step
This tweet is potentially suicidal.


In [26]:
# Let's say we have a new tweet
new_tweet = "Sometime its better to end ourselves"

# Preprocess the tweet
new_tweet = preprocess_text(new_tweet)

# Tokenize and pad the tweet
new_tweet = tokenizer.texts_to_sequences([new_tweet])
new_tweet = pad_sequences(new_tweet, maxlen=100)

# Use the model to predict the sentiment of the tweet
prediction = model.predict(new_tweet)

# Print the prediction
if prediction > 0.5:
    print("This tweet is potentially suicidal.")
else:
    print("This tweet is not suicidal.")


1/1 [==============================] - 0s 30ms/step
This tweet is potentially suicidal.


In [27]:
# Let's say we have a new tweet
new_tweet = "I apolgised to my girlfriend for cheating but she is not forgiving me. I have lost the hope that things will ever get better bwtween us. It's so frustating me. I can't imagine my life without her"

# Preprocess the tweet
new_tweet = preprocess_text(new_tweet)

# Tokenize and pad the tweet
new_tweet = tokenizer.texts_to_sequences([new_tweet])
new_tweet = pad_sequences(new_tweet, maxlen=100)

# Use the model to predict the sentiment of the tweet
prediction = model.predict(new_tweet)

# Print the prediction
if prediction > 0.5:
    print("This tweet is potentially suicidal.")
else:
    print("This tweet is not suicidal.")

1/1 [==============================] - 0s 26ms/step
This tweet is potentially suicidal.


In [28]:
# Let's say we have a new tweet
new_tweet = "She came in my life and gave me hope to live"

# Preprocess the tweet
new_tweet = preprocess_text(new_tweet)

# Tokenize and pad the tweet
new_tweet = tokenizer.texts_to_sequences([new_tweet])
new_tweet = pad_sequences(new_tweet, maxlen=100)

# Use the model to predict the sentiment of the tweet
prediction = model.predict(new_tweet)

# Print the prediction
if prediction > 0.5:
    print("This tweet is potentially suicidal.")
else:
    print("This tweet is not suicidal.")

1/1 [==============================] - 0s 29ms/step
This tweet is not suicidal.
